In [ ]:
%%bash

pip install --upgrade 'scikit-learn<1.6.0'

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Frame the Problem (15 pts)

Answer the following questions.

1. Define the objective: what are you trying to predict?
2. How should you frame this problem as a machine learning problem?
3. How should performance be measured? 


1. Define the objective: What are we trying to predict?

The goal is to predict the selling price of a house based on its different features. The dataset includes details like the size of the house and it's location, the age, and quality of the house, all of these we’re going to use to estimate the final sale price.

2. How should you frame this problem as a machine learning problem?

This is a supervised regression problem because the target (SalePrice) is continuous. We’re going to train the model on data that already includes known sale prices, so it can learn how each feature affects the outcome and then predict prices for unseen houses.

3. How do we measure performance?

The evaluation metric is Root Mean Squared Logarithmic Error (RMSLE). It compares the log of predicted prices to the log of actual prices and penalizes large errors more than small ones. Getting a lower RMSLE means it perfomed better. 

# 2. Get the Data (15 pts)

Minimum expectations:

1. Load the data.
2. Check the size and type of data.

In [ ]:
import pandas as pd

# 1. Load training and test data
train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

# 2. Check the size and type of data
print("Train shape:", train.shape)
print("Test shape:", test.shape)

train.info()

# 3. Explore the Data (15 pts)

General expectations

1. Discuss the types of features (categorical, ordinal, int/float, bounded/unbounded, text, etc.)
2. What features have missing values?
3. Are there any outliers? If so, how can you tell?
4. What features appear usefulness for the task? Are there any that appear useless? Why?
5. Identify and characterize the target distribution.
6. Visualize the data.
7. Study the correlations between attributes.


In [ ]:
import matplotlib.pyplot as plt

# 1. Feature types
num_cols = train.select_dtypes(include=['number']).columns.tolist()
cat_cols = train.select_dtypes(exclude=['number']).columns.tolist()
print("Numeric features:", len(num_cols))
print("Categorical features:", len(cat_cols))

# 2. Missing values
na = train.isna().sum()
na = na[na > 0].sort_values(ascending=False)
print("\nMissing values (non-zero):")
print(na)

# 3. Outliers
plt.figure()
plt.boxplot(train["SalePrice"].dropna(), vert=False)
plt.title("SalePrice — boxplot")
plt.show()

# 4. Useful features
corr_to_target = train[num_cols].corrwith(train["SalePrice"]).sort_values(ascending=False)
print("\nTop correlations with SalePrice:")
print(corr_to_target.head(10))

# 5. Target distribution
plt.figure()
plt.hist(train["SalePrice"].dropna(), bins=30)
plt.title("SalePrice — distribution")
plt.xlabel("SalePrice"); plt.ylabel("Count")
plt.show()

# 6&7. Correlations between attributes
top10 = corr_to_target.head(10).index
print("\nCorrelation matrix among top 10 features (rounded):")
print(train[top10].corr().round(2))


### Step 3: Explore the Data

1. Types of features:  
   The dataset contains 38 numeric features and 43 categorical ones. Numeric columns include things like 'LotArea', 'GrLivArea', and 'SalePrice', while categorical ones describe property attributes such as 'Neighborhood', 'HouseStyle', and 'GarageType'.

2. Missing values: 
   Several columns have missing data. The highest counts are in 'PoolQC', 'MiscFeature', 'Alley', and 'Fence', which makes sense because not all houses have pools, alleys, or fences. Some garage- and basement-related columns also have missing values for homes without those features.

3. Outliers:  
   The boxplot of 'SalePrice' shows a few extreme values on the right side, meaning there are some very expensive houses compared to the rest. These points represent outliers but are valid observations rather than errors.

4. Useful features: 
   Based on correlation, 'OverallQual', 'GrLivArea', and 'GarageCars' have the strongest positive relationships with 'SalePrice'. This suggests that overall house quality, size, and garage capacity are important predictors.

5. Target distribution:  
   The histogram of 'SalePrice' shows that the prices are right-skewed — most houses are moderately priced, with a few high-end properties increasing the tail.

6. Visualizations:  
   The scatter plot between 'GrLivArea' and 'SalePrice' shows a clear upward trend: larger houses usually sell for more, though a few large houses sell for less than expected (possible outliers).

7. Correlations between attributes:  
   The correlation matrix shows strong relationships among size-related variables ('GrLivArea', 'TotalBsmtSF', 'GarageArea', '1stFlrSF'). This means these features overlap in the information they provide and may need attention later to avoid redundancy.


# 4. Prepare the Data for ML (15 pts)

Use the Scikit Learn data pipeline and preprocessing API. Write functions for any custom data transformations used that are not available in the Scikit Learn API.

Minimum expectations. Some of these may or may not be necessary for this task. If a step is not necessary, discuss why.

1. Clean the data.
2. Perform feature selection.
3. Perform feature engineering.
5. Standardize or normalize features.

Your data preparation pipeline does not need to use all of the features, but you must use at least one categorical feature, one ordinal feature, and one numeric feature. These can be raw features, or features that you create yourself.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# 1. Clean the data
# We can fill in all the missing values
num_features = ["GrLivArea"]            # numeric
cat_features = ["Neighborhood"]         # categorical
ord_features = ["OverallQual"]          # ordinal (already numeric but treated separately)

# numeric: fill with median
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# categorical: fill with most frequent, then one-hot encode
cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# ordinal: fill with median (values already ranked 1–10)
ord_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# combine all transformations
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, num_features),
        ("cat", cat_transformer, cat_features),
        ("ord", ord_transformer, ord_features)
    ]
)

# define X and y
X = train[num_features + cat_features + ord_features]
y = train["SalePrice"]

# split into train/test sets
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

# fit transform the training data
X_train_prepared = preprocessor.fit_transform(X_train)
X_valid_prepared = preprocessor.transform(X_valid)

print("Shape after preprocessing:", X_train_prepared.shape)


# 5. Explore Different Models (15 pts)

Minimum expectations:

1. Train an unregularized linear model, a linear model with default parameters, and a decision tree.
2. Measure and compare their performance.
3. Analyze the most significant variables for each algorithm.
4. Analyze the types of errors the models make.


In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# 1. Train an unregularized linear model, a linear model with default parameters, and a decision tree.
lin_reg = LinearRegression()
ridge_reg = Ridge(random_state=42)
tree_reg = DecisionTreeRegressor(random_state=42)

models = {
    "Linear Regression": lin_reg,
    "Ridge Regression": ridge_reg,
    "Decision Tree": tree_reg
}

results = {}
for name, model in models.items():
    model.fit(X_train_prepared, y_train)
    preds = model.predict(X_valid_prepared)
    rmse = np.sqrt(mean_squared_error(y_valid, preds))
    r2 = r2_score(y_valid, preds)
    results[name] = {"RMSE": rmse, "R2": r2}

# 2. Measure and compare their performance
print("Model Performance Comparison:")
for name, score in results.items():
    print(f"{name:20s} | RMSE: {score['RMSE']:.2f} | R²: {score['R2']:.3f}")

# 3. Analyze the most significant variables for each algorithm (for linear models)
print("\nTop 5 features (Linear Regression):")
lin_coefs = lin_reg.coef_.flatten()
top_idx = np.argsort(np.abs(lin_coefs))[-5:][::-1]
print(top_idx, "→ coefficients with largest absolute values")

# 4. Analyze types of errors
best_model = min(results, key=lambda x: results[x]['RMSE'])
print(f"\nModel with lowest error: {best_model}")


### Step 5: Explore Different Models

1. Models trained: 
   I tested three models:  
   - Linear Regression (no regularization)  
   - Ridge Regression (regularized linear model)  
   - Decision Tree Regressor  

2. Performance comparison:
   The results show that both Linear and Ridge Regression performed very similarly with RMSE ≈ 38,500 and R² ≈ 0.806, while the Decision Tree had a slightly higher error and lower R² ≈ 0.786.  
   → The best model overall was 'Linear Regression', giving the lowest RMSE.

3. Most significant variables: 
   The top coefficients from the Linear Regression model came from the encoded 'Neighborhood' features and 'GrLivArea', showing that house size and neighborhood have the largest effect on price prediction.

4. Error analysis:  
   The linear models slightly underpredict high-value homes because of the skewed target distribution, while the Decision Tree tends to overfit and produce larger errors for outliers.


# 6. Fine-tune the Models (15 pts)

1. Fine-tune the hyperparameters for a linear model using RandomizedSearchCV.
2. Fine-tune the hyperparameters for a decision tree using GridSearchCV.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# 1. Fine-tune Ridge (linear model) using RandomizedSearchCV
ridge = Ridge(random_state=42)
ridge_params = {"alpha": [0.01, 0.1, 1, 10, 100]}
ridge_search = RandomizedSearchCV(ridge, ridge_params, cv=5, scoring="neg_root_mean_squared_error", n_iter=5, random_state=42)
ridge_search.fit(X_train_prepared, y_train)
ridge_best = ridge_search.best_estimator_
ridge_rmse = np.sqrt(mean_squared_error(y_valid, ridge_best.predict(X_valid_prepared)))

print("Best Ridge params:", ridge_search.best_params_)
print("Ridge validation RMSE:", ridge_rmse)

# 2. Fine-tune Decision Tree using GridSearchCV
tree = DecisionTreeRegressor(random_state=42)
tree_params = {"max_depth": [None, 5, 10], "min_samples_split": [2, 5, 10]}
tree_search = GridSearchCV(tree, tree_params, cv=5, scoring="neg_root_mean_squared_error")
tree_search.fit(X_train_prepared, y_train)
tree_best = tree_search.best_estimator_
tree_rmse = np.sqrt(mean_squared_error(y_valid, tree_best.predict(X_valid_prepared)))

print("\nBest Decision Tree params:", tree_search.best_params_)
print("Decision Tree validation RMSE:", tree_rmse)

### Step 6: Fine-tune the Models

Ridge performed best with **alpha = 1** (RMSE ≈ 38,551).  
The Decision Tree performed best with **max_depth = 5** and **min_samples_split = 2** (RMSE ≈ 35,279).  
Tuning slightly improved the Decision Tree’s accuracy.


# 7. Summarize your results (15 pts)

1. Provide a 2-3 sentence summary of your solution for a non-technical audience of real estate executives.

2. Provide a technical summary of your solution for the ML team lead at a real estate investment company.

### Step 7: Summarize Your Results

**Non-technical summary:**  
The model predicts house prices based on features like overall quality, size, and neighborhood. After testing different models, the Decision Tree gave the most accurate predictions with around a 35K average error.

**Technical summary:**  
Linear and Ridge Regression performed similarly (R² ≈ 0.81), while the tuned Decision Tree achieved the lowest RMSE (~35,279). The preprocessing pipeline used one numeric, one categorical, and one ordinal feature with scaling and encoding to prepare data for training.


# 9. Submit your predictions to the competition

In [ ]:
X_test = test[num_features + cat_features + ord_features]
X_test_prepared = preprocessor.transform(X_test)

# Predict using the fine-tuned Decision Tree
test_predictions = tree_best.predict(X_test_prepared)

In [ ]:
submission_df = (
    pd.read_csv(
        "/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv",
        index_col="Id"
    ).assign(
        SalePrice=test_predictions
    ).to_csv(
        "submission.csv",
        index=True,
    )
)


In [ ]:
%%bash

cat submission.csv | head -n 5